# Modelo Serving - Vehículos Eléctricos Colombia

Entrenamiento, registro en MLflow/Unity Catalog y despliegue de modelo predictivo de adopción de vehículos eléctricos/híbridos. Combina un modelo de regresión (RandomForest) para predecir cantidad de eléctricos por departamento y un modelo de crecimiento logístico para la tendencia nacional.

In [0]:
# ============================================================
# CELL 1: IMPORTS Y CARGA DE DATOS DESDE UNITY CATALOG
# ============================================================
import pandas as pd
import numpy as np
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

# Scipy
from scipy.optimize import curve_fit

# MLflow
import mlflow
from mlflow.models import infer_signature

print("="*70)
print("📊 CARGA DE DATOS DESDE UNITY CATALOG")
print("="*70)

# Cargar tablas Gold desde Unity Catalog
df_dep_spark = spark.table("workspace.default.gold_proporcion_electricos_departamento")
df_anio_spark = spark.table("workspace.default.gold_proporcion_electricos_anio")

# Convertir a pandas
df_dep = df_dep_spark.toPandas()
df_anio = df_anio_spark.toPandas()

print(f"\n✅ gold_proporcion_electricos_departamento: {len(df_dep)} filas")
print(f"   Columnas: {list(df_dep.columns)}")
print(f"   Años: {sorted(df_dep['REGISTRO'].unique())}")
print(f"   Departamentos: {df_dep['DEPARTAMENTO'].nunique()}")

print(f"\n✅ gold_proporcion_electricos_anio: {len(df_anio)} filas")
print(f"   Columnas: {list(df_anio.columns)}")

print("\n📊 Muestra de datos por departamento:")
display(df_dep.head(10))

print("\n📊 Datos nacionales por año:")
display(df_anio)

📊 CARGA DE DATOS DESDE UNITY CATALOG

✅ gold_proporcion_electricos_departamento: 384 filas
   Columnas: ['DEPARTAMENTO', 'REGISTRO', 'TOTAL_PARQUE', 'TOTAL_ELECTRICOS', 'PROPORCION_PORCENTAJE']
   Años: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   Departamentos: 32

✅ gold_proporcion_electricos_anio: 12 filas
   Columnas: ['REGISTRO', 'TOTAL_PARQUE', 'TOTAL_ELECTRICOS', 'PROPORCION_PORCENTAJE']

📊 Muestra de datos por departamento:


DEPARTAMENTO,REGISTRO,TOTAL_PARQUE,TOTAL_ELECTRICOS,PROPORCION_PORCENTAJE
AMAZONAS,2015,565,0,0.0
AMAZONAS,2016,942,0,0.0
AMAZONAS,2017,1415,0,0.0
AMAZONAS,2018,1101,5,0.4541
AMAZONAS,2019,1296,1,0.0772
AMAZONAS,2020,905,0,0.0
AMAZONAS,2021,950,0,0.0
AMAZONAS,2022,1551,0,0.0
AMAZONAS,2023,1377,3,0.2179
AMAZONAS,2024,1183,0,0.0



📊 Datos nacionales por año:


REGISTRO,TOTAL_PARQUE,TOTAL_ELECTRICOS,PROPORCION_PORCENTAJE
2015,961147,586,0.061
2016,840462,504,0.06
2017,746453,450,0.0603
2018,811804,2104,0.2592
2019,878388,5435,0.6187
2020,719950,7908,1.0984
2021,998830,20304,2.0328
2022,1094000,30513,2.7891
2023,892434,34039,3.8142
2024,1043520,53934,5.1685


In [0]:
# ============================================================
# CELL 2: FEATURE ENGINEERING
# ============================================================
print("="*70)
print("🔧 FEATURE ENGINEERING")
print("="*70)

# Separar features y target
X_raw = df_dep[['REGISTRO', 'DEPARTAMENTO', 'TOTAL_PARQUE']].copy()
y = df_dep['TOTAL_ELECTRICOS'].copy()

# Renombrar columnas para el modelo
X_raw.columns = ['year', 'department', 'total_parque']

# One-Hot Encoding para DEPARTAMENTO
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
dept_encoded = encoder.fit_transform(X_raw[['department']])

# Nombres de columnas one-hot
dept_columns = [f"dept_{c}" for c in encoder.categories_[0]]

# DataFrame con features codificadas
X_encoded = pd.DataFrame(dept_encoded, columns=dept_columns, index=X_raw.index)
X_encoded['year'] = X_raw['year'].astype(float)
X_encoded['total_parque'] = X_raw['total_parque'].astype(float)

# Reordenar: year, total_parque, dept_*
feature_cols = ['year', 'total_parque'] + dept_columns
X = X_encoded[feature_cols]

print(f"\nFeatures totales: {X.shape[1]}")
print(f"  - year: 1")
print(f"  - total_parque: 1")
print(f"  - department one-hot: {len(dept_columns)}")

# Split temporal: entrenamiento 2015-2023, prueba 2024-2026
train_mask = X_raw['year'] <= 2023
test_mask = X_raw['year'] >= 2024

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"\n📦 Train: {len(X_train)} filas (2015-2023)")
print(f"📦 Test:  {len(X_test)} filas (2024-2026)")

# Guardar encoder y features como artifacts
os.makedirs("/tmp/model_artifacts", exist_ok=True)
with open("/tmp/model_artifacts/encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)
with open("/tmp/model_artifacts/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("\n✅ Encoder y feature_cols guardados en /tmp/model_artifacts/")

🔧 FEATURE ENGINEERING

Features totales: 34
  - year: 1
  - total_parque: 1
  - department one-hot: 32

📦 Train: 288 filas (2015-2023)
📦 Test:  96 filas (2024-2026)

✅ Encoder y feature_cols guardados en /tmp/model_artifacts/


In [0]:
# ============================================================
# CELL 3: ENTRENAMIENTO DEL MODELO DE CONTEO
# ============================================================
print("="*70)
print("🤖 ENTRENAMIENTO - Modelo de Conteo (RandomForest)")
print("="*70)

count_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

count_model.fit(X_train, y_train)

# Predicciones
y_pred_train = count_model.predict(X_train)
y_pred_test = count_model.predict(X_test)

# Métricas
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"\n📊 MÉTRICAS DEL MODELO:")
print(f"   Train - R²: {r2_train:.4f} | RMSE: {rmse_train:.2f}")
print(f"   Test  - R²: {r2_test:.4f} | RMSE: {rmse_test:.2f}")

# Comparar predicciones vs reales (test)
df_comparison = pd.DataFrame({
    'year': X_test['year'].values.astype(int),
    'department': X_raw.loc[test_mask, 'department'].values,
    'total_parque': X_test['total_parque'].values.astype(int),
    'real_electricos': y_test.values,
    'pred_electricos': y_pred_test.astype(int),
    'real_proporcion_pct': (y_test.values / X_test['total_parque'].values * 100).round(2),
    'pred_proporcion_pct': (y_pred_test / X_test['total_parque'].values * 100).round(2)
})

print("\n📊 Comparación predicciones vs reales (Test 2024-2026):")
display(df_comparison.head(20))

# Guardar modelo como artifact
with open("/tmp/model_artifacts/count_model.pkl", "wb") as f:
    pickle.dump(count_model, f)
print("\n✅ Modelo guardado en /tmp/model_artifacts/count_model.pkl")

🤖 ENTRENAMIENTO - Modelo de Conteo (RandomForest)

📊 MÉTRICAS DEL MODELO:
   Train - R²: 0.8994 | RMSE: 450.13
   Test  - R²: 0.4127 | RMSE: 5065.34

📊 Comparación predicciones vs reales (Test 2024-2026):


year,department,total_parque,real_electricos,pred_electricos,real_proporcion_pct,pred_proporcion_pct
2024,AMAZONAS,1183,0,1,0.0,0.11
2025,AMAZONAS,1758,0,2,0.0,0.12
2026,AMAZONAS,1596,3,1,0.19,0.09
2024,ANTIOQUIA,169709,9224,4998,5.44,2.95
2025,ANTIOQUIA,228407,15879,4927,6.95,2.16
2026,ANTIOQUIA,201323,18584,4927,9.23,2.45
2024,ARAUCA,5726,9,6,0.16,0.11
2025,ARAUCA,6717,5,9,0.07,0.14
2026,ARAUCA,4886,4,6,0.08,0.13
2024,"ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA",1277,6,2,0.47,0.21



✅ Modelo guardado en /tmp/model_artifacts/count_model.pkl


In [0]:
# ============================================================
# CELL 4: MODELO DE CRECIMIENTO LOGÍSTICO (Nacional)
# ============================================================
print("="*70)
print("📈 MODELO DE CRECIMIENTO LOGÍSTICO - Tendencia Nacional")
print("="*70)

# Función logística: P(t) = K / (1 + exp(-r*(t-t0)))
def logistic(x, K, r, t0):
    return K / (1 + np.exp(-r * (x - t0)))

# Datos nacionales
x_nacional = df_anio['REGISTRO'].values.astype(float)
y_nacional = df_anio['PROPORCION_PORCENTAJE'].values

# Ajustar modelo logístico
popt, pcov = curve_fit(logistic, x_nacional, y_nacional, p0=[50, 0.5, 2022], maxfev=10000)
K, r, t0 = popt

# R² del modelo logístico
y_pred_log = logistic(x_nacional, *popt)
ss_res = np.sum((y_nacional - y_pred_log) ** 2)
ss_tot = np.sum((y_nacional - np.mean(y_nacional)) ** 2)
r2_log = 1 - (ss_res / ss_tot)

print(f"\n📊 Parámetros del modelo logístico:")
print(f"   Techo de adopción (K): {K:.2f}%")
print(f"   Tasa de crecimiento (r): {r:.4f}")
print(f"   Punto de inflexión (t0): año {t0:.0f}")
print(f"   R² = {r2_log:.4f}")

# Proyección 2015-2035
years_proj = np.arange(2015, 2036)
prop_proj = logistic(years_proj, *popt)
df_proj = pd.DataFrame({
    'año': years_proj,
    'proporcion_logistica_pct': np.round(prop_proj, 2)
})

print("\n📈 Proyección nacional de adopción (logístico):")
display(df_proj)

# Guardar parámetros logísticos
logistic_params = {"K": float(K), "r": float(r), "t0": float(t0), "r2": float(r2_log)}
with open("/tmp/model_artifacts/logistic_params.json", "w") as f:
    json.dump(logistic_params, f, indent=2)
print(f"\n✅ Parámetros guardados en /tmp/model_artifacts/logistic_params.json")

📈 MODELO DE CRECIMIENTO LOGÍSTICO - Tendencia Nacional

📊 Parámetros del modelo logístico:
   Techo de adopción (K): 9.51%
   Tasa de crecimiento (r): 0.5512
   Punto de inflexión (t0): año 2024
   R² = 0.9982

📈 Proyección nacional de adopción (logístico):


año,proporcion_logistica_pct
2015,0.08
2016,0.14
2017,0.24
2018,0.4
2019,0.68
2020,1.12
2021,1.79
2022,2.73
2023,3.91
2024,5.21



✅ Parámetros guardados en /tmp/model_artifacts/logistic_params.json


In [0]:
# ============================================================
# CELL 5: CLASE PythonModel PERSONALIZADA
# ============================================================
print("="*70)
print("🏗️ DEFINICIÓN DEL PythonModel")
print("="*70)

class ModeloVehiculosElectricos(mlflow.pyfunc.PythonModel):
    """
    Modelo personalizado que combina:
    1. RandomForest para predecir cantidad de eléctricos por departamento
    2. Modelo logístico para tendencia nacional de proporción
    """
    
    def load_context(self, context):
        """Carga los modelos desde los artifacts."""
        with open(context.artifacts["encoder"], "rb") as f:
            self.encoder = pickle.load(f)
        with open(context.artifacts["count_model"], "rb") as f:
            self.count_model = pickle.load(f)
        with open(context.artifacts["feature_cols"], "r") as f:
            self.feature_cols = json.load(f)
        with open(context.artifacts["logistic_params"], "r") as f:
            self.logistic_params = json.load(f)
    
    def _logistic(self, x, K, r, t0):
        return K / (1 + np.exp(-r * (x - t0)))
    
    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Predice cantidad y proporción de vehículos eléctricos/híbridos.
        
        Args:
            model_input: DataFrame con columnas 'year', 'department', 'total_parque'
        
        Returns:
            DataFrame con: predicted_electricos, predicted_proporcion_pct, logistic_proportion_pct
        """
        df = model_input.copy()
        
        # One-hot encode department
        dept_encoded = self.encoder.transform(df[['department']])
        dept_columns = [f"dept_{c}" for c in self.encoder.categories_[0]]
        df_encoded = pd.DataFrame(dept_encoded, columns=dept_columns, index=df.index)
        df_encoded['year'] = df['year'].values.astype(float)
        df_encoded['total_parque'] = df['total_parque'].values.astype(float)
        
        # Reordenar columnas
        X = df_encoded[self.feature_cols]
        
        # Predecir conteo
        predicted_electricos = self.count_model.predict(X)
        predicted_electricos = np.maximum(predicted_electricos, 0).astype(int)
        
        # Calcular proporción (evitar división por cero)
        total_parque = df['total_parque'].values
        predicted_proporcion = np.where(
            total_parque > 0,
            (predicted_electricos / total_parque) * 100,
            0.0
        )
        
        # Proyección logística nacional
        K = self.logistic_params['K']
        r = self.logistic_params['r']
        t0 = self.logistic_params['t0']
        logistic_prop = self._logistic(df['year'].values, K, r, t0)
        
        return pd.DataFrame({
            'predicted_electricos': predicted_electricos,
            'predicted_proporcion_pct': np.round(predicted_proporcion, 4),
            'logistic_proportion_pct': np.round(logistic_prop, 4)
        })

print("✅ Clase ModeloVehiculosElectricos definida")
print("   Input:  DataFrame[year, department, total_parque]")
print("   Output: DataFrame[predicted_electricos, predicted_proporcion_pct, logistic_proportion_pct]")

🏗️ DEFINICIÓN DEL PythonModel
✅ Clase ModeloVehiculosElectricos definida
   Input:  DataFrame[year, department, total_parque]
   Output: DataFrame[predicted_electricos, predicted_proporcion_pct, logistic_proportion_pct]


In [0]:
# ============================================================
# CELL 6: LOG EN MLFLOW Y REGISTRO EN UNITY CATALOG
# ============================================================
print("="*70)
print("📝 REGISTRO EN MLFLOW Y UNITY CATALOG")
print("="*70)

# Configurar MLflow con Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/modelo_vehiculos_electricos")

# Input example
input_example = pd.DataFrame({
    'year': [2027, 2028, 2029],
    'department': ['BOGOTA D.C.', 'ANTIOQUIA', 'VALLE DEL CAUCA'],
    'total_parque': [900000, 2100000, 1300000]
})

# Generar output example directamente (sin instanciar PythonModel)
dept_encoded_ex = encoder.transform(input_example[['department']])
dept_cols_ex = [f"dept_{c}" for c in encoder.categories_[0]]
df_enc_ex = pd.DataFrame(dept_encoded_ex, columns=dept_cols_ex, index=input_example.index)
df_enc_ex['year'] = input_example['year'].values.astype(float)
df_enc_ex['total_parque'] = input_example['total_parque'].values.astype(float)
X_ex = df_enc_ex[feature_cols]

pred_elec_ex = count_model.predict(X_ex)
pred_elec_ex = np.maximum(pred_elec_ex, 0).astype(int)
pred_prop_ex = (pred_elec_ex / input_example['total_parque'].values) * 100
log_prop_ex = logistic(input_example['year'].values, *popt)

output_example = pd.DataFrame({
    'predicted_electricos': pred_elec_ex,
    'predicted_proporcion_pct': np.round(pred_prop_ex, 4),
    'logistic_proportion_pct': np.round(log_prop_ex, 4)
})

# Inferir signature
signature = infer_signature(input_example, output_example)
print(f"\n📊 Signature inferida:")
print(f"   Input:  {list(input_example.columns)}")
print(f"   Output: {list(output_example.columns)}")

# Artifacts
artifacts = {
    "encoder": "/tmp/model_artifacts/encoder.pkl",
    "count_model": "/tmp/model_artifacts/count_model.pkl",
    "feature_cols": "/tmp/model_artifacts/feature_cols.json",
    "logistic_params": "/tmp/model_artifacts/logistic_params.json"
}

registered_model_name = "workspace.default.modelo_vehiculos_electricos"

with mlflow.start_run(run_name="modelo_vehiculos_electricos_v1") as run:
    # Log métricas
    mlflow.log_metric("r2_test", r2_test)
    mlflow.log_metric("rmse_test", float(rmse_test))
    mlflow.log_metric("r2_logistic", r2_log)
    mlflow.log_param("model_type", "RandomForest + Logistic")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("train_years", "2015-2023")
    mlflow.log_param("test_years", "2024-2026")
    
    # Log modelo
    model_info = mlflow.pyfunc.log_model(
        name="modelo_vehiculos_electricos",
        python_model=ModeloVehiculosElectricos(),
        artifacts=artifacts,
        signature=signature,
        input_example=input_example,
        pip_requirements=["scikit-learn", "pandas", "numpy", "scipy"],
    )
    print(f"\n✅ Modelo loggeado en MLflow: {model_info.model_uri}")
    
    # Registrar en Unity Catalog
    registered_version = mlflow.register_model(
        model_uri=model_info.model_uri,
        name=registered_model_name,
        await_registration_for=300,
    )
    print(f"✅ Registrado en UC: {registered_model_name} v{registered_version.version}")
    
    # Establecer alias "Champion"
    from mlflow import MlflowClient
    registry_client = MlflowClient(registry_uri="databricks-uc")
    registry_client.set_registered_model_alias(
        name=registered_model_name,
        alias="Champion",
        version=registered_version.version,
    )
    print(f"✅ Alias 'Champion' asignado a versión {registered_version.version}")

print(f"\n📋 Modelo disponible en: models:/{registered_model_name}@Champion")

📝 REGISTRO EN MLFLOW Y UNITY CATALOG

📊 Signature inferida:
   Input:  ['year', 'department', 'total_parque']
   Output: ['predicted_electricos', 'predicted_proporcion_pct', 'logistic_proportion_pct']


🔗 View Logged Model at: https://dbc-9af4a986-b999.cloud.databricks.com/ml/experiments/1979900867036129/models/m-77d43ed215404c659f6c6a1d94f13758?o=7474650402665980
2026/09/19 19:16:24 INFO mlflow.pyfunc: Validating input example against model signature



✅ Modelo loggeado en MLflow: models:/m-77d43ed215404c659f6c6a1d94f13758


Successfully registered model 'workspace.default.modelo_vehiculos_electricos'.


Uploading artifacts:   0%|          | 0/16 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.modelo_vehiculos_electricos': https://dbc-9af4a986-b999.cloud.databricks.com/explore/data/models/workspace/default/modelo_vehiculos_electricos/version/1?o=7474650402665980


✅ Registrado en UC: workspace.default.modelo_vehiculos_electricos v1
✅ Alias 'Champion' asignado a versión 1

📋 Modelo disponible en: models:/workspace.default.modelo_vehiculos_electricos@Champion


In [0]:
# ============================================================
# CELL 7: TEST DEL MODELO REGISTRADO
# ============================================================
print("="*70)
print("🧪 TEST DEL MODELO REGISTRADO")
print("="*70)

# Cargar modelo desde Unity Catalog usando alias
model_uri = f"models:/{registered_model_name}@Champion"
loaded_model = mlflow.pyfunc.load_model(model_uri)
print(f"✅ Modelo cargado desde: {model_uri}")

# Datos de prueba: 2027-2029 para BOGOTA D.C., ANTIOQUIA, VALLE DEL CAUCA
test_data = pd.DataFrame({
    'year': [2027, 2027, 2027, 2028, 2028, 2028, 2029, 2029, 2029],
    'department': [
        'BOGOTA D.C.', 'ANTIOQUIA', 'VALLE DEL CAUCA',
        'BOGOTA D.C.', 'ANTIOQUIA', 'VALLE DEL CAUCA',
        'BOGOTA D.C.', 'ANTIOQUIA', 'VALLE DEL CAUCA'
    ],
    'total_parque': [
        950000, 2200000, 1350000,
        1000000, 2300000, 1400000,
        1050000, 2400000, 1450000
    ]
})

# Hacer predicciones
predictions = loaded_model.predict(test_data)
results = pd.concat([test_data, predictions], axis=1)

print("\n📊 PREDICCIONES 2027-2029:")
display(results)

print("\n📋 Resumen por departamento:")
for dept in ['BOGOTA D.C.', 'ANTIOQUIA', 'VALLE DEL CAUCA']:
    dept_data = results[results['department'] == dept]
    print(f"\n  {dept}:")
    for _, row in dept_data.iterrows():
        print(f"    {int(row['year'])}: {row['predicted_electricos']:,} eléctricos "
              f"({row['predicted_proporcion_pct']:.2f}%) | Logístico nacional: {row['logistic_proportion_pct']:.2f}%")

🧪 TEST DEL MODELO REGISTRADO


✅ Modelo cargado desde: models:/workspace.default.modelo_vehiculos_electricos@Champion

📊 PREDICCIONES 2027-2029:


year,department,total_parque,predicted_electricos,predicted_proporcion_pct,logistic_proportion_pct
2027,BOGOTA D.C.,950000,7880,0.8295,8.2142
2027,ANTIOQUIA,2200000,4927,0.224,8.2142
2027,VALLE DEL CAUCA,1350000,4688,0.3473,8.2142
2028,BOGOTA D.C.,1000000,7880,0.788,8.7187
2028,ANTIOQUIA,2300000,4927,0.2142,8.7187
2028,VALLE DEL CAUCA,1400000,4688,0.3349,8.7187
2029,BOGOTA D.C.,1050000,7880,0.7505,9.0387
2029,ANTIOQUIA,2400000,4927,0.2053,9.0387
2029,VALLE DEL CAUCA,1450000,4688,0.3233,9.0387



📋 Resumen por departamento:

  BOGOTA D.C.:
    2027: 7,880 eléctricos (0.83%) | Logístico nacional: 8.21%
    2028: 7,880 eléctricos (0.79%) | Logístico nacional: 8.72%
    2029: 7,880 eléctricos (0.75%) | Logístico nacional: 9.04%

  ANTIOQUIA:
    2027: 4,927 eléctricos (0.22%) | Logístico nacional: 8.21%
    2028: 4,927 eléctricos (0.21%) | Logístico nacional: 8.72%
    2029: 4,927 eléctricos (0.21%) | Logístico nacional: 9.04%

  VALLE DEL CAUCA:
    2027: 4,688 eléctricos (0.35%) | Logístico nacional: 8.21%
    2028: 4,688 eléctricos (0.33%) | Logístico nacional: 8.72%
    2029: 4,688 eléctricos (0.32%) | Logístico nacional: 9.04%


In [0]:
# ============================================================
# CELL 8: INSTRUCCIONES PARA MODEL SERVING ENDPOINT
# ============================================================
print("="*70)
print("🚀 INSTRUCCIONES PARA CREAR ENDPOINT DE SERVING")
print("="*70)

print(f"""
📋 MODELO REGISTRADO:
   Nombre: {registered_model_name}
   Alias:  Champion
   URI:    models:/{registered_model_name}@Champion
   Versión: {registered_version.version}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1️⃣ CREAR ENDPOINT VÍA UI:
   a. Ve a Serving en el sidebar de Databricks
   b. Click en "Create serving endpoint"
   c. Nombre: modelo-vehiculos-electricos
   d. Selecciona el modelo: {registered_model_name}
   e. Versión: {registered_version.version} (o alias: Champion)
   f. Workload type: CPU → Size: Small
   g. Click "Create"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2️⃣ CREAR ENDPOINT VÍA DATABRICKS CLI:

   cat > /tmp/endpoint_config.json << 'EOF'
   {{
     "name": "modelo-vehiculos-electricos",
     "config": {{
       "served_entities": [
         {{
           "entity_name": "{registered_model_name}",
           "entity_version": "{registered_version.version}",
           "workload_type": "CPU",
           "workload_size": "Small",
           "scale_to_zero_enabled": true
         }}
       ]
     }}
   }}
   EOF

   databricks serving-endpoints create --json-file /tmp/endpoint_config.json
   databricks serving-endpoints get modelo-vehiculos-electricos
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3️⃣ CREAR ENDPOINT VÍA PYTHON SDK:

   from databricks.sdk import WorkspaceClient
   from databricks.sdk.service.serving import (
       EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType
   )

   w = WorkspaceClient()
   w.serving_endpoints.create(
       name="modelo-vehiculos-electricos",
       config=EndpointCoreConfigInput(
           served_entities=[ServedEntityInput(
               entity_name="{registered_model_name}",
               entity_version="{registered_version.version}",
               workload_type=ServingModelWorkloadType.CPU,
               workload_size="Small",
               scale_to_zero_enabled=True,
           )]
       )
   )
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4️⃣ PROBAR EL ENDPOINT CON CURL:

   curl -X POST \\
     "https://<DATABRICKS_URL>/api/2.0/serving-endpoints/modelo-vehiculos-electricos/invocations" \\
     -H "Authorization: Bearer <TOKEN>" \\
     -H "Content-Type: application/json" \\
     -d '{{
       "inputs": [
         {{"year": 2027, "department": "BOGOTA D.C.", "total_parque": 950000}},
         {{"year": 2028, "department": "ANTIOQUIA", "total_parque": 2300000}},
         {{"year": 2029, "department": "VALLE DEL CAUCA", "total_parque": 1450000}}
       ]
     }}'
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5️⃣ PROBAR VÍA PYTHON:

   import requests, json

   url = "https://<WORKSPACE>.cloud.databricks.com/api/2.0/serving-endpoints/modelo-vehiculos-electricos/invocations"
   headers = {{"Authorization": "Bearer <TOKEN>", "Content-Type": "application/json"}}
   data = {{"inputs": [
       {{"year": 2027, "department": "BOGOTA D.C.", "total_parque": 950000}},
       {{"year": 2028, "department": "ANTIOQUIA", "total_parque": 2300000}}
   ]}}
   resp = requests.post(url, headers=headers, json=data)
   print(json.dumps(resp.json(), indent=2))
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 NOTAS:
   • El endpoint tarda ~15 min en estar ready la primera vez
   • scale_to_zero_enabled=True ahorra costos sin tráfico (cold start en primer request)
   • Para producción: considera scale_to_zero_enabled=False
   • Monitorea latencia y throughput en la pestaña Metrics del endpoint
   • Conecta Power BI vía Databricks connector para consumir las tablas Gold directamente
""")

🚀 INSTRUCCIONES PARA CREAR ENDPOINT DE SERVING

📋 MODELO REGISTRADO:
   Nombre: workspace.default.modelo_vehiculos_electricos
   Alias:  Champion
   URI:    models:/workspace.default.modelo_vehiculos_electricos@Champion
   Versión: 1

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1️⃣ CREAR ENDPOINT VÍA UI:
   a. Ve a Serving en el sidebar de Databricks
   b. Click en "Create serving endpoint"
   c. Nombre: modelo-vehiculos-electricos
   d. Selecciona el modelo: workspace.default.modelo_vehiculos_electricos
   e. Versión: 1 (o alias: Champion)
   f. Workload type: CPU → Size: Small
   g. Click "Create"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2️⃣ CREAR ENDPOINT VÍA DATABRICKS CLI:

   cat > /tmp/endpoint_config.json << 'EOF'
   {
     "name": "modelo-vehiculos-electricos",
     "config": {
       "served_entities": [
         {
           "entity_name": "workspace.default.modelo_vehiculos_electricos",
           "entit